In [1]:
# =========================================
# 05B_build_doi_aggregated_table.ipynb
# Build DOI-level aggregated regression table
# =========================================
import os
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)

WORK_DIR = r"C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026"

MASTER_FILE = os.path.join(WORK_DIR, "interim", "master_table.parquet")
X_FILE = os.path.join(WORK_DIR, "interim", "X_features.parquet")
Y_FILE = os.path.join(WORK_DIR, "interim", "y_target.parquet")
Y_RAW_FILE = os.path.join(WORK_DIR, "interim", "y_target_raw.parquet")
GROUP_FILE = os.path.join(WORK_DIR, "interim", "groups.parquet")
ROWID_FILE = os.path.join(WORK_DIR, "interim", "row_ids.parquet")

INTERIM_DIR = os.path.join(WORK_DIR, "interim")
REPORT_DIR = os.path.join(WORK_DIR, "reports")
META_DIR = os.path.join(WORK_DIR, "metadata")
ARTIFACT_DIR = os.path.join(WORK_DIR, "artifacts", "05B_doi_aggregation")

for p in [INTERIM_DIR, REPORT_DIR, META_DIR, ARTIFACT_DIR]:
    os.makedirs(p, exist_ok=True)

print("MASTER_FILE:", MASTER_FILE)
print("X_FILE:", X_FILE)
print("Y_FILE:", Y_FILE)
print("GROUP_FILE:", GROUP_FILE)

MASTER_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\master_table.parquet
X_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\X_features.parquet
Y_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\y_target.parquet
GROUP_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\groups.parquet


In [2]:
master = pd.read_parquet(MASTER_FILE)
X = pd.read_parquet(X_FILE)
y = pd.read_parquet(Y_FILE)["T80_log1p"].copy()
y_raw = pd.read_parquet(Y_RAW_FILE)["T80_clean"].copy()
groups = pd.read_parquet(GROUP_FILE)["Ref_DOI_number"].copy()
row_ids = pd.read_parquet(ROWID_FILE)["raw_row_id"].copy()

assert len(master) == len(X) == len(y) == len(y_raw) == len(groups) == len(row_ids)

print("master shape:", master.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("unique DOI:", groups.nunique())

master shape: (1835, 69)
X shape: (1835, 77)
y shape: (1835,)
unique DOI: 964


In [3]:
device_df = X.copy()
device_df["T80_log1p"] = y.values
device_df["T80_clean"] = y_raw.values
device_df["Ref_DOI_number"] = groups.values
device_df["raw_row_id"] = row_ids.values

# optional metadata from master
meta_cols = [c for c in ["publication_year"] if c in master.columns]
for c in meta_cols:
    device_df[c] = master[c].values

print("device_df shape:", device_df.shape)
device_df.head(2)

device_df shape: (1835, 82)


,is_nip,is_pin,is_other_arch,etl_has_tio2,etl_has_sno2,etl_has_pcbm,etl_has_c60,etl_has_zno,htl_has_spiro,htl_has_ptaa,htl_has_pedot,htl_has_niox,htl_has_p3ht,back_has_au,back_has_ag,back_has_al,back_has_carbon,has_perovskite_additives,has_etl_additives,has_htl_additives,band_gap_ev,perovskite_thickness_nm,etl_thickness_nm,cell_area_measured_cm2,n_cells_per_substrate,encapsulation_flag,protocol_has_l,protocol_has_d,bias_is_mpp,bias_is_oc,bias_is_sc,light_intensity_suns,is_dark_condition,is_approx_1sun,is_high_light,temperature_c,rh_pct,is_room_temperature,is_hot_test,is_dry_condition,is_humid_condition,architecture_family_nip,architecture_family_other_or_unknown,architecture_family_pin,etl_family_c60,etl_family_other_or_unknown,etl_family_other_rare,etl_family_pcbm,etl_family_sno2,etl_family_tio2,etl_family_zno,htl_family_missing,htl_family_niox,htl_family_other_or_unknown,htl_family_other_rare,htl_family_p3ht,htl_family_pedot_pss,htl_family_ptaa,htl_family_spiro_ometad,backcontact_family_ag,backcontact_family_al,backcontact_family_au,backcontact_family_carbon,backcontact_family_cu,backcontact_family_other_rare,protocol_family_isos_d,protocol_family_isos_l,protocol_family_other_isos,protocol_family_other_or_unknown,protocol_family_other_rare,bias_family_mpp,bias_family_open_circuit,bias_family_other_rare,light_bin_dark_or_zero,light_bin_high_light,light_bin_missing,light_bin_other_rare,T80_log1p,T80_clean,Ref_DOI_number,raw_row_id,publication_year
0,1,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,1.59,NaN,NaN,0.06,0.0,0.0,1,0,1,0,0,100.0,0,0,1,25.0,NaN,1,0,0,0,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,5.303305,200.0,10.1039/c9ta01893j,27,2019
1,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,NaN,500.0,25.0,0.16,0.0,0.0,1,0,1,0,0,100.0,0,0,1,25.0,NaN,1,0,0,0,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,5.017280,150.0,10.1002/aenm.201803587,45,2019


In [4]:
feature_cols = [c for c in X.columns]

numeric_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c in feature_cols]

print("n feature cols:", len(feature_cols))
print("n numeric/bool feature cols:", len(numeric_cols))
print("sample numeric cols:", numeric_cols[:20])

n feature cols: 77
n numeric/bool feature cols: 77
sample numeric cols: ['is_nip', 'is_pin', 'is_other_arch', 'etl_has_tio2', 'etl_has_sno2', 'etl_has_pcbm', 'etl_has_c60', 'etl_has_zno', 'htl_has_spiro', 'htl_has_ptaa', 'htl_has_pedot', 'htl_has_niox', 'htl_has_p3ht', 'back_has_au', 'back_has_ag', 'back_has_al', 'back_has_carbon', 'has_perovskite_additives', 'has_etl_additives', 'has_htl_additives']


In [5]:
agg_dict = {}

# aggregate all numeric / bool encoded features by mean
# for one-hot columns, mean = fraction within DOI
for c in numeric_cols:
    agg_dict[c] = "mean"

# targets
agg_dict["T80_log1p"] = "median"
agg_dict["T80_clean"] = "median"

# metadata
if "publication_year" in device_df.columns:
    agg_dict["publication_year"] = "median"

doi_table = (
    device_df
    .groupby("Ref_DOI_number", as_index=False)
    .agg(agg_dict)
)

# add DOI-level sample count
doi_counts = (
    device_df.groupby("Ref_DOI_number")
    .size()
    .rename("n_devices_in_doi")
    .reset_index()
)

doi_table = doi_table.merge(doi_counts, on="Ref_DOI_number", how="left")

print("doi_table shape:", doi_table.shape)
print("unique DOI rows:", doi_table["Ref_DOI_number"].nunique())
doi_table.head(3)

doi_table shape: (964, 82)
unique DOI rows: 964


,Ref_DOI_number,is_nip,is_pin,is_other_arch,etl_has_tio2,etl_has_sno2,etl_has_pcbm,etl_has_c60,etl_has_zno,htl_has_spiro,htl_has_ptaa,htl_has_pedot,htl_has_niox,htl_has_p3ht,back_has_au,back_has_ag,back_has_al,back_has_carbon,has_perovskite_additives,has_etl_additives,has_htl_additives,band_gap_ev,perovskite_thickness_nm,etl_thickness_nm,cell_area_measured_cm2,n_cells_per_substrate,encapsulation_flag,protocol_has_l,protocol_has_d,bias_is_mpp,bias_is_oc,bias_is_sc,light_intensity_suns,is_dark_condition,is_approx_1sun,is_high_light,temperature_c,rh_pct,is_room_temperature,is_hot_test,is_dry_condition,is_humid_condition,architecture_family_nip,architecture_family_other_or_unknown,architecture_family_pin,etl_family_c60,etl_family_other_or_unknown,etl_family_other_rare,etl_family_pcbm,etl_family_sno2,etl_family_tio2,etl_family_zno,htl_family_missing,htl_family_niox,htl_family_other_or_unknown,htl_family_other_rare,htl_family_p3ht,htl_family_pedot_pss,htl_family_ptaa,htl_family_spiro_ometad,backcontact_family_ag,backcontact_family_al,backcontact_family_au,backcontact_family_carbon,backcontact_family_cu,backcontact_family_other_rare,protocol_family_isos_d,protocol_family_isos_l,protocol_family_other_isos,protocol_family_other_or_unknown,protocol_family_other_rare,bias_family_mpp,bias_family_open_circuit,bias_family_other_rare,light_bin_dark_or_zero,light_bin_high_light,light_bin_missing,light_bin_other_rare,T80_log1p,T80_clean,publication_year,n_devices_in_doi
0,10.1002/adfm.201504245,1.0,0.0,0.0,1.0,0.000000,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.55,250.0,100.0,0.10,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,27.0,30.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,1.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,4.693490,115.5,2016.0,2
1,10.1002/adfm.201600910,1.0,0.0,0.0,0.0,0.666667,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.59,NaN,NaN,0.09,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,20.0,70.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.333333,0.0,0.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.945910,6.0,2016.0,3
2,10.1002/adfm.201605988,1.0,0.0,0.0,1.0,0.000000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.5,1.0,1.0,1.48,NaN,NaN,0.10,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,25.0,55.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,4.141494,65.0,2017.0,2


In [6]:
assert doi_table["Ref_DOI_number"].isna().sum() == 0
assert doi_table["T80_log1p"].isna().sum() == 0
assert doi_table["T80_clean"].isna().sum() == 0

print("DOI rows:", len(doi_table))
print("Min devices/DOI:", doi_table["n_devices_in_doi"].min())
print("Median devices/DOI:", doi_table["n_devices_in_doi"].median())
print("Max devices/DOI:", doi_table["n_devices_in_doi"].max())

DOI rows: 964
Min devices/DOI: 1
Median devices/DOI: 1.0
Max devices/DOI: 12


In [7]:
TARGET_COL = "T80_log1p"
RAW_TARGET_COL = "T80_clean"
GROUP_COL = "Ref_DOI_number"

forbidden_cols = [TARGET_COL, RAW_TARGET_COL, GROUP_COL]

doi_X = doi_table.drop(columns=forbidden_cols, errors="ignore").copy()
doi_y = doi_table[TARGET_COL].copy()
doi_groups = doi_table[GROUP_COL].copy()

print("doi_X shape:", doi_X.shape)
print("doi_y shape:", doi_y.shape)
print("doi_groups shape:", doi_groups.shape)
doi_X.head(2)

doi_X shape: (964, 79)
doi_y shape: (964,)
doi_groups shape: (964,)


,is_nip,is_pin,is_other_arch,etl_has_tio2,etl_has_sno2,etl_has_pcbm,etl_has_c60,etl_has_zno,htl_has_spiro,htl_has_ptaa,htl_has_pedot,htl_has_niox,htl_has_p3ht,back_has_au,back_has_ag,back_has_al,back_has_carbon,has_perovskite_additives,has_etl_additives,has_htl_additives,band_gap_ev,perovskite_thickness_nm,etl_thickness_nm,cell_area_measured_cm2,n_cells_per_substrate,encapsulation_flag,protocol_has_l,protocol_has_d,bias_is_mpp,bias_is_oc,bias_is_sc,light_intensity_suns,is_dark_condition,is_approx_1sun,is_high_light,temperature_c,rh_pct,is_room_temperature,is_hot_test,is_dry_condition,is_humid_condition,architecture_family_nip,architecture_family_other_or_unknown,architecture_family_pin,etl_family_c60,etl_family_other_or_unknown,etl_family_other_rare,etl_family_pcbm,etl_family_sno2,etl_family_tio2,etl_family_zno,htl_family_missing,htl_family_niox,htl_family_other_or_unknown,htl_family_other_rare,htl_family_p3ht,htl_family_pedot_pss,htl_family_ptaa,htl_family_spiro_ometad,backcontact_family_ag,backcontact_family_al,backcontact_family_au,backcontact_family_carbon,backcontact_family_cu,backcontact_family_other_rare,protocol_family_isos_d,protocol_family_isos_l,protocol_family_other_isos,protocol_family_other_or_unknown,protocol_family_other_rare,bias_family_mpp,bias_family_open_circuit,bias_family_other_rare,light_bin_dark_or_zero,light_bin_high_light,light_bin_missing,light_bin_other_rare,publication_year,n_devices_in_doi
0,1.0,0.0,0.0,1.0,0.000000,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.55,250.0,100.0,0.10,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,27.0,30.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,1.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,2016.0,2
1,1.0,0.0,0.0,0.0,0.666667,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.59,NaN,NaN,0.09,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,20.0,70.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.333333,0.0,0.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,2016.0,3


In [8]:
fully_missing_cols = [c for c in doi_X.columns if doi_X[c].isna().all()]
doi_X = doi_X.drop(columns=fully_missing_cols, errors="ignore").copy()

constant_cols = [c for c in doi_X.columns if doi_X[c].nunique(dropna=True) <= 1]
doi_X = doi_X.drop(columns=constant_cols, errors="ignore").copy()

print("Dropped fully missing:", fully_missing_cols)
print("Dropped constant:", constant_cols)
print("Final doi_X shape:", doi_X.shape)

Dropped fully missing: []
Dropped constant: []
Final doi_X shape: (964, 79)


In [9]:
DOI_TABLE_OUT = os.path.join(INTERIM_DIR, "05B_doi_table.parquet")
DOI_X_OUT = os.path.join(INTERIM_DIR, "05B_doi_X.parquet")
DOI_Y_OUT = os.path.join(INTERIM_DIR, "05B_doi_y.parquet")
DOI_G_OUT = os.path.join(INTERIM_DIR, "05B_doi_groups.parquet")

doi_table.to_parquet(DOI_TABLE_OUT, index=False)
doi_X.to_parquet(DOI_X_OUT, index=False)
pd.DataFrame({"T80_log1p": doi_y}).to_parquet(DOI_Y_OUT, index=False)
pd.DataFrame({"Ref_DOI_number": doi_groups}).to_parquet(DOI_G_OUT, index=False)

print("Saved:", DOI_TABLE_OUT)
print("Saved:", DOI_X_OUT)
print("Saved:", DOI_Y_OUT)
print("Saved:", DOI_G_OUT)

Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\05B_doi_table.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\05B_doi_X.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\05B_doi_y.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\05B_doi_groups.parquet


In [10]:
summary = {
    "n_device_rows_input": int(len(device_df)),
    "n_doi_rows_output": int(len(doi_table)),
    "n_features_output": int(doi_X.shape[1]),
    "median_devices_per_doi": float(doi_table["n_devices_in_doi"].median()),
    "mean_devices_per_doi": float(doi_table["n_devices_in_doi"].mean()),
}

pd.DataFrame([summary]).to_csv(
    os.path.join(REPORT_DIR, "05B_doi_aggregation_summary.csv"),
    index=False
)

card = {
    "notebook": "05B_build_doi_aggregated_table.ipynb",
    "inputs": {
        "master": MASTER_FILE,
        "X": X_FILE,
        "y": Y_FILE,
        "y_raw": Y_RAW_FILE,
        "groups": GROUP_FILE,
    },
    "outputs": {
        "doi_table": DOI_TABLE_OUT,
        "doi_X": DOI_X_OUT,
        "doi_y": DOI_Y_OUT,
        "doi_groups": DOI_G_OUT,
    },
    "summary": summary,
    "aggregation_rules": {
        "encoded_features": "mean within DOI",
        "T80_log1p": "median within DOI",
        "T80_clean": "median within DOI",
        "publication_year": "median within DOI if available",
        "n_devices_in_doi": "count within DOI",
    }
}

card_path = os.path.join(META_DIR, "05B_doi_aggregation_card.json")
with open(card_path, "w", encoding="utf-8") as f:
    json.dump(card, f, indent=4)

print("Saved:", card_path)
print("DONE.")

Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\metadata\05B_doi_aggregation_card.json
DONE.
